In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import pickle

np.random.seed(42)
if np.random.choice(np.arange(1000)) != 102:
    raise ValueError("Random seed is not set correctly.")

# 1 Choose Dataset

In [ ]:
datasets = ['ml-1m', 'steam', 'goodreads', 'ml-10m']
DATASET = datasets[3]
print(f"Dataset: {DATASET}")

base_artifacts = Path.cwd().parents[1] / 'CausalI2I_artifacts'

Dataset: ml-10m


# 2 Load Dataset

In [3]:
raw_data_path = base_artifacts / 'Datasets' / 'Raw' / DATASET

if DATASET == 'ml-1m':
    ratings_columns = ['user_id', 'item_id', 'rank', 'timestamp']
    data = pd.read_csv(
        raw_data_path / 'ratings.dat',
        sep='::', 
        names=ratings_columns, 
        engine='python')

    item_data = pd.read_csv(
        raw_data_path / 'movies.dat',
        sep='::', 
        names=['item_id', 'title', 'genre'], 
        engine='python', 
        encoding='iso-8859-1')
    
    get_item_name = {item_id: item_data[item_data['item_id'] == item_id]['title'].values[0] for item_id in item_data['item_id']}
    data['item_id'] = data['item_id'].map(get_item_name)

if DATASET == 'steam':
    data = pd.read_csv(
        raw_data_path / 'steam_filtered.csv'
    )
    data.columns = ['user_id', 'item_id', 'timestamp']

if DATASET == 'goodreads':
    data = pd.read_csv(
        raw_data_path / f'goodreads_filtered.csv'
    )
    data.columns = ['user_id', 'item_id', 'timestamp']

if DATASET == 'ml-10m':
    data = pd.read_csv(
        raw_data_path / 'processed_ml-10m.csv'
    )
    data.columns = ['user_id', 'item_id', 'timestamp']

data = data.sort_values(by=['user_id', 'timestamp']).reset_index(drop=True)

In [4]:
unique_users = data['user_id'].unique()
unique_items = data['item_id'].unique()

n_users = len(unique_users)
n_items = len(unique_items)

print(f"Number of unique users: {n_users:,}")
item_series = data.groupby('item_id').size()
print(f"Numbers of users per items:")
print(f"\t Min: {item_series.min():,}")
print(f"\t 5th: {item_series.quantile(0.05):,.2f}")
print(f"\t Median: {item_series.median():,}")
print(f"\t Mean: {item_series.mean():,.2f}")
print(f"\t 95th: {item_series.quantile(0.95):,.2f}")
print(f"\t Max: {item_series.max():,}")

print(f"\nNumber of unique items: {n_items:,}")
user_series = data.groupby('user_id').size()
print(f"Numbers of items per users:")
print(f"\t Min: {user_series.min():,}")
print(f"\t 5th: {user_series.quantile(0.05):,.2f}")
print(f"\t Median: {user_series.median():,}")
print(f"\t Mean: {user_series.mean():,.2f}")
print(f"\t 95th: {user_series.quantile(0.95):,.2f}")
print(f"\t Max: {user_series.max():,}")


Number of unique users: 5,369
Numbers of users per items:
	 Min: 84
	 5th: 245.00
	 Median: 671.0
	 Mean: 1,008.96
	 95th: 3,040.85
	 Max: 4,817

Number of unique items: 3,362
Numbers of items per users:
	 Min: 194
	 5th: 380.00
	 Median: 540.0
	 Mean: 631.80
	 95th: 1,190.80
	 Max: 3,095


# 3 Rename Entries

In [5]:
old2new_users = {old: new for new, old in enumerate(unique_users)}
old2new_items = {old: new for new, old in enumerate(unique_items)}
item_dict = {new: old for new, old in enumerate(unique_items)}
title2id = {v: k for k, v in item_dict.items()}

data['user_id'] = data['user_id'].map(old2new_users)
data['item_id'] = data['item_id'].map(old2new_items)

In [6]:
data_path = base_artifacts / 'Datasets' / 'Processed' / DATASET

# Save the processed data
data.to_csv(data_path / 'data_clean.csv', index=False)

# Save the item dictionary
with open(data_path / 'item_dict.pkl', 'wb') as f:
    pickle.dump(item_dict, f)

# Save a random user test set
rng = np.random.default_rng(seed=42)
test_users = rng.choice(np.arange(n_users), size=int(0.2 * n_users), replace=False)
with open(data_path / 'test_users.pkl', 'wb') as f:
    pickle.dump(test_users, f)

# 4 Choose 10K Pairs

In [8]:
data_pairs = data.copy()
data_pairs['interaction'] = 1
pivot_table = data_pairs.pivot(index='user_id', columns='item_id', values='interaction').fillna(0)

In [9]:
X = pivot_table.values
mean = X.mean(axis=0)
std  = np.maximum(X.std(axis=0, ddof=1), 1e-8)
M = (X - mean) / std
corr_mat = (M.T @ M) / (M.shape[0] - 1)
np.fill_diagonal(corr_mat, 0)

In [10]:
chosen_pairs_flat_coordinates = corr_mat.flatten().argsort()[::-1][:10000]
chosen_pairs_ids = [(i % n_items, i // n_items) for i in chosen_pairs_flat_coordinates]
chosen_pairs_titles = [(item_dict[i], item_dict[j]) for i, j in chosen_pairs_ids]

In [12]:
chosen_pairs_path = base_artifacts / 'Chosen_Pairs' / DATASET

with open(chosen_pairs_path / 'chosen_pairs_ids.pkl', 'wb') as f:
    pickle.dump(chosen_pairs_ids, f)
    
with open(chosen_pairs_path / 'chosen_pairs_titles.pkl', 'wb') as f:
    pickle.dump(chosen_pairs_titles, f)